# BinX AI & ML Internship
## Week 5 — Day 4: t-SNE & Anomaly Detection

**Topics covered:**
- t-SNE: a visualization-only alternative to PCA that preserves local neighborhoods
- PCA vs t-SNE: when to use each and what each reveals
- What anomaly detection is and why it matters
- Isolation Forest: the algorithm, the contamination parameter, and its output
- Connection between DBSCAN noise points (Day 2) and anomaly detection

---
## 1. Where I'm Starting From

By the end of Day 3, I had applied PCA to the insurance dataset and reduced it to 9 components (7 for 95% variance, 2 for visualization).

PCA works by finding the axes of *maximum global variance*. That makes it good for compression — but the 2D scatter it produces can be "blurry": clusters that are actually separate in the original space may overlap in the PCA projection.

Today I learn two things:

**1. t-SNE** — a dimensionality reduction technique built specifically for visualization. It doesn't preserve global structure; it preserves *local neighborhoods*. The result is usually much cleaner cluster separation in 2D.

**2. Anomaly Detection (Isolation Forest)** — finding data points that are significantly different from the rest, without any labels.

```text
Day 1: K-Means     → grouped similar points
Day 2: DBSCAN      → labeled some points as noise (outliers)
Day 3: PCA         → compressed 9 features to 2 for visualization
Day 4: t-SNE       → better visualization of those same clusters
       Isolation Forest → formally identifies and quantifies the outliers
```

---
## 2. What t-SNE Is

**t-SNE** stands for *t-distributed Stochastic Neighbor Embedding*.

The name is long, but the idea is simple:

```text
For every point in high-dimensional space:
  → find its nearest neighbors
  → map it to 2D so that those neighbors stay close

Points that were close together in high-D → stay close in 2D
Points that were far apart in high-D       → stay far in 2D
```

This makes t-SNE excellent at revealing clusters visually, even when they are not linearly separable — which is when PCA's flat projection fails.

---

### The key parameter: perplexity

`perplexity` controls how many neighbors each point "pays attention to" during the embedding:

```text
Low perplexity (5–10):   focuses on very tight local structure
                         → many small, tight clusters

Medium perplexity (30):  balanced — the standard starting point

High perplexity (50+):   focuses on broader neighborhoods
                         → fewer, more spread-out clusters
```

For most datasets: start with `perplexity=30`.

---

### The sklearn API

```python
from sklearn.manifold import TSNE

tsne   = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)   # shape: (n_samples, 2)

# Note: TSNE has no .transform() — it must be refit for any new data
```

---

### Critical limitation: the axes are meaningless

```text
In PCA:
  PC1 = 0.65 × charges + 0.58 × smoker + ...
  → The axis has a clear meaning I can interpret

In t-SNE:
  Dimension 1: no formula, no interpretation
  → The axes rotate differently every run (even with same random_state)
  → Only the relative position of points matters
```

**t-SNE is for looking, not for modelling.**
Its output should never be fed into a downstream model — use PCA or original features for that.

---
## 3. PCA vs t-SNE — Side-by-Side

| | PCA | t-SNE |
|---|---|---|
| **What it preserves** | Global variance (the big spread) | Local neighborhoods (who is close to whom) |
| **Main use** | Compression + visualization | Visualization only |
| **Speed** | Very fast | Slow on large datasets |
| **Axes meaning** | Interpretable weighted combinations | No meaning — only relative position |
| **Use output in model?** | Yes (as reduced features) | No |
| **Cluster separation** | Sometimes blurry | Usually much cleaner |
| **Deterministic?** | Yes | Approximate (random_state fixes it) |

---

### When to choose which

```text
Use PCA when:
  → You need to compress features for a model
  → You want to know which original features matter
  → You have a large dataset and speed matters

Use t-SNE when:
  → You want the clearest possible 2D visualization
  → You want to see whether clusters actually exist
  → You're doing exploratory work, not modelling
```

---
## 4. What Anomaly Detection Is

**Anomaly detection** finds data points that differ significantly from the norm.

Examples:
- A credit card transaction that is 10× the usual amount → fraud
- A machine sensor reading that spikes above normal range → hardware failure
- A medical record with impossible values → data entry error
- An insurance claim with extreme charges and unusual feature combination → potential fraud

---

### Why unsupervised?

Anomalies are rare by definition. In most real datasets:

```text
Normal points:  thousands of labeled examples
Anomalies:      maybe 10, maybe 0 — often unlabeled
```

This makes supervised classification difficult (too few positive examples).
Unsupervised methods learn what "normal" looks like and flag whatever deviates.

---

### Connection to Day 2

DBSCAN's noise points (labeled -1) are already a form of anomaly detection:
points that don't fit any dense cluster are flagged as outliers.

Isolation Forest is a dedicated anomaly detection algorithm — more rigorous and with a tunable threshold.

---
## 5. Isolation Forest

**Isolation Forest** is based on a clever observation:

```text
Normal points sit inside a dense cluster.
To isolate a normal point, many random splits are needed.

Anomalous points sit far from the dense mass.
To isolate an anomaly, very few splits are needed.

→ Points that are easy to isolate = anomalies
```

The algorithm builds many random decision trees, each randomly splitting the data.
For each point, it measures the average number of splits needed to isolate it across all trees.
Short average path length → anomaly.

---

### The contamination parameter

`contamination` is the only thing I need to set:

```python
IsolationForest(contamination=0.05)
# → Flag the 5% of points with the shortest isolation paths as anomalies
```

It is my estimate of what fraction of the dataset is anomalous.
If I don't know, starting with 0.05 (5%) is a reasonable default.

---

### The API

```python
from sklearn.ensemble import IsolationForest

iso   = IsolationForest(contamination=0.05, random_state=42)
preds = iso.fit_predict(X_scaled)
# Output: -1 for anomaly, +1 for normal

scores = iso.decision_function(X_scaled)
# Lower score → more anomalous
# Useful for ranking: which anomalies are the most extreme?
```

---

### Summary

```text
t-SNE        → best 2D visualization tool; preserves local neighborhoods
             → axes meaningless; visualization only, not for models

Anomaly det. → unsupervised; learns "normal" and flags deviations

Isolation Forest → random partitioning; easy-to-isolate = anomaly
                → contamination sets the expected fraction of anomalies
                → output: -1 (anomaly) or +1 (normal)
```

The practical application is in `Hands_On_Lab.ipynb`.